# 💥 Failure Modes

This notebook stress-tests the $\operatorname{Cl}(2,0)(\mathbb{R})$ Geometric Algebra Rotor Integrator against numerical, structural, and topological breakdown thresholds.

## Agenda
1. **Geometric Denormalization:**
2. **Spurious High-Frequency Energy Pumping (Aliasing):**
3. **Coordinate Singularity Blindspots:**
4. **Phase Space Distortion (Integrator-Induced Dissipation):**

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

from kuramoto.analysis.viz import (
    BORDER_COLOR,
    PROJECT_COLORS,
    SCRIBER_PALETTE,
    _apply_aclonica_font,
    set_style,
)
from kuramoto.metrics.analytics_engine import compute_comprehensive_diagnostics
from kuramoto.solvers.rotor import RotorSolver, step_rkmk4

# Apply project dark-mode aesthetics
set_style()


@dataclass(frozen=True)
class Config:
    """Single source of truth for simulation parameters.

    Frozen deliberately: rebinding a field raises AttributeError instead of
    silently leaking a different N into downstream cells.
    """

    n_osc: int = 20           # canonical oscillator count
    n_osc_plot: int = 10      # reduced count, display-only (Mode 3 circle plot)
    seed: int = 27
    K_active: float = 0.5
    t_max: float = 100.0
    dt: float = 0.005


CONFIG = Config()
print("🚀 Environment initialized with Clifford rotor engine, visual theme, and canonical Config.")

## 1️⃣ Failure Mode 1: Geometric Denormalization

In [ ]:
print("👟 Running Failure Mode 1: Manifold Drift Tracking (Long-Horizon)...")

# Parameters sourced strictly from CONFIG (SHARED-1)
steps_long = int(CONFIG.t_max / CONFIG.dt)
times_long = np.linspace(0.0, CONFIG.t_max, steps_long + 1)

# Instantiate un-normalized tracker
solver_f1 = RotorSolver(n_oscillators=CONFIG.n_osc, dim=2, seed=CONFIG.seed)
manifold_drift = np.zeros(steps_long + 1)

# Map initial state magnitude (SC-8: scalar^2 + bivector^2 in Cl(2,0))
def compute_rotor_norm_drift(rotor_array):
    if np.iscomplexobj(rotor_array):
        return np.max(np.abs(np.abs(rotor_array)**2 - 1.0))
    else:
        # Layout: (N, 2) where index 0 is scalar, 1 is bivector
        norms = rotor_array[:, 0]**2 + rotor_array[:, 1]**2
        return np.max(np.abs(norms - 1.0))

manifold_drift[0] = compute_rotor_norm_drift(solver_f1._rotor_array)

# Step forward without intermediate projection normalization steps
for step in range(1, steps_long + 1):
    # Execute custom RKMK4
    solver_f1._rotor_array = step_rkmk4(solver_f1, CONFIG.K_active, CONFIG.dt)

    # TODO(FM-1): Ambiguous whether Mode 1 measures intrinsic RKMK4 drift or injected float corruption.
    # Commented-out perturbation block preserved awaiting decision D-1 from Renzo.
    # if step % 100 == 0:
    #     solver_f1._rotor_array *= 1.0 + 1e-9 * np.random.randn(*solver_f1._rotor_array.shape)

    manifold_drift[step] = compute_rotor_norm_drift(solver_f1._rotor_array)

print(f"    [Mode 1 Max Manifold Drift]: {manifold_drift.max():.3e}")

# Plot results
fig, axes1 = plt.subplots(figsize=(8, 5))

axes1.plot(times_long, manifold_drift, color=SCRIBER_PALETTE["accent_blue"], lw=2.0)
axes1.axhline(1e-6, color='magenta', linestyle='--', alpha=0.7, label='Machine Epsilon Bound')
axes1.set_yscale('log')
axes1.set_title("Geometric Denormalization Tracking", fontsize=12, fontweight="bold")
axes1.set_xlabel("Simulation Time (s)", fontsize=11)
axes1.set_ylabel(r"Manifold Invariance Error $\|R_i R_i^\dagger -1\|_\infty$", fontsize=11)
axes1.legend()
axes1.grid(True, which="both", alpha=0.1)
_apply_aclonica_font(axes1)
plt.tight_layout()
plt.show()

## 2️⃣ Failure Mode 2: Spurious High-Frequency Energy Pumping (Aliasing)

In [ ]:
print("👟 Running Failure Mode 2: Step-Size Sensitivity Sweep (PSD Floors)...")

# Fast sampling vs ultra-fast sampling comparison (SHARED-1)
dt_base = 0.01
dt_half = 0.005
t_psd = 10.0

# Base step trajectory
solver_base = RotorSolver(n_oscillators=CONFIG.n_osc, dim=2, seed=CONFIG.seed)
_, phases_base = solver_base.simulate(K=CONFIG.K_active, t_eval=np.arange(0, t_psd, dt_base), dt_internal=dt_base, method="geometric")
v_base = np.gradient(phases_base, dt_base, axis=0)
freqs_base = np.abs(np.fft.rfftfreq(v_base.shape[0], d=dt_base))
psd_base = np.abs(np.fft.rfft(v_base, axis=0))**2
mean_psd_base = np.mean(psd_base, axis=1)

# Halved step trajectory
solver_half = RotorSolver(n_oscillators=CONFIG.n_osc, dim=2, seed=CONFIG.seed)
_, phases_half = solver_half.simulate(K=CONFIG.K_active, t_eval=np.arange(0, t_psd, dt_half), dt_internal=dt_half, method="geometric")
v_half = np.gradient(phases_half, dt_half, axis=0)
freqs_half = np.abs(np.fft.rfftfreq(v_half.shape[0], d=dt_half))
psd_half = np.abs(np.fft.rfft(v_half, axis=0))**2
mean_psd_half = np.mean(psd_half, axis=1)

# FM-2: Per-array frequency band masks (each spectrum has its own Nyquist limit)
band_base = freqs_base > 0.25 * freqs_base.max()
band_half = freqs_half > 0.25 * freqs_half.max()

mean_psd_base_upper = np.mean(mean_psd_base[band_base])
mean_psd_half_upper = np.mean(mean_psd_half[band_half])
alias_ratio = float(mean_psd_base_upper / max(mean_psd_half_upper, 1e-30))

# FM-3: Oscillator-wise maximum PSD reporting across frequency bins
max_psd_base = np.max(psd_base, axis=1)
max_psd_half = np.max(psd_half, axis=1)
alias_ratio_max = float(
    np.mean(max_psd_base[band_base]) / max(np.mean(max_psd_half[band_half]), 1e-30)
)

print(f"    [Aliasing Ratio (Mean PSD Upper Band)]:  {alias_ratio:.4f}")
print(f"    [Aliasing Ratio (Worst-Case Max PSD)]:   {alias_ratio_max:.4f}")

# Plot results
fig, axes2 = plt.subplots(figsize=(8, 5))
axes2.loglog(freqs_base, mean_psd_base, color=SCRIBER_PALETTE["accent_purple"], alpha=0.8, label=r"Base Step $\Delta t$")
axes2.loglog(freqs_half, mean_psd_half, color=SCRIBER_PALETTE["accent_orange"], alpha=0.6, label=r"Halved Step $\Delta t / 2$")
axes2.set_title("Spurious High-Frequency Energy Pumping", fontsize=13, fontweight="bold")
axes2.set_xlabel("Frequency (Hz)", fontsize=11)
axes2.set_ylabel("Power Spectral Density (PSD)", fontsize=11)
axes2.legend()
axes2.grid(True, alpha=0.1)
_apply_aclonica_font(axes2)

plt.tight_layout()
plt.show()

## 3️⃣ Failure Mode 3: Coordinate Singularity Blindspots

In [ ]:
print("👟 Running Failure Mode 3: Chaotic Phase Boundary Stress Test...")

# Initialize uniform entropic noise over [-pi, pi] (SHARED-1: CONFIG.n_osc)
seed_chaos = 1227
rng = np.random.default_rng(seed_chaos)
init_phases_chaos = rng.uniform(-np.pi, np.pi, CONFIG.n_osc)

solver_chaos = RotorSolver(n_oscillators=CONFIG.n_osc, dim=2, seed=seed_chaos)
# Override internal state to ensure cross-boundary drift conditions (SC-8: rotor sign convention)
if np.iscomplexobj(solver_chaos._rotor_array):
    solver_chaos._rotor_array = np.cos(init_phases_chaos/2.0) - 1j * np.sin(init_phases_chaos/2.0)
else:
    solver_chaos._rotor_array[:, 0] = np.cos(init_phases_chaos/2.0)
    solver_chaos._rotor_array[:, 1] = np.sin(init_phases_chaos/2.0)

# FM-4: Guard Mode 3 variables before try block (pessimistic default: assume failure until proven otherwise)
has_nan_failure = True
max_velocity_spike = np.inf

# Track if any NaN spikes occur near coupling floor boundary K -> 0
try:
    _, simulated_phases_chaos = solver_chaos.simulate(K=1e-5, t_eval=np.linspace(0, 20, 2000), dt_internal=0.01, method="geometric")
    nan_count = int(np.sum(np.isnan(simulated_phases_chaos)))
    has_nan_failure = nan_count > 0
    # FM-5: Unwrap phases before differentiating to eliminate 2pi/dt branch cut jump artifacts
    unwrapped_chaos = np.unwrap(simulated_phases_chaos, axis=0)
    max_velocity_spike = float(np.max(np.abs(np.gradient(unwrapped_chaos, 0.01, axis=0))))
    print(f"    [Chaos Check] NaN encountered: {has_nan_failure} (count={nan_count}) | Max Vel Spike (unwrapped): {max_velocity_spike:.3f} rad/s")
except Exception as e:
    print(f"❌ Coordinate system mapping threw an explicit boundary failure exception: {e}")

# Plot results (FM-6: derive t_slice dynamically)
fig, axes3 = plt.subplots(figsize=(8, 5))

from matplotlib.collections import LineCollection

t_slice = min(200, simulated_phases_chaos.shape[0])  # FM-6: derived slice length
n_plot = CONFIG.n_osc_plot  # SHARED-1: reduced display-only oscillator count for visual clarity in circle plot

# Slice trajectories: shape (T, N_plot)
phases_to_plot = simulated_phases_chaos[:t_slice, :n_plot]

# Compute explicit spatial coordinates on the unit circle
X = np.cos(phases_to_plot)
Y = np.sin(phases_to_plot)

# Compute dynamic global order parameter vector R(t)
r_complex_trajectory = np.mean(np.exp(1j * phases_to_plot), axis=1)
r_mag = np.abs(r_complex_trajectory)
r_angle = np.angle(r_complex_trajectory)

# 1. Draw static geometric reference background (The Unit Circle Manifold)
circle_angles = np.linspace(0, 2 * np.pi, 300)
axes3.plot(np.cos(circle_angles), np.sin(circle_angles), color="#2A2A2A", linestyle="-", lw=2.0, zorder=1)
axes3.axhline(0, color="#2A2A2A", lw=1.0, linestyle=":", zorder=1)
axes3.axvline(0, color="#2A2A2A", lw=1.0, linestyle=":", zorder=1)

# 2. Select distinct base colors for each displayed oscillator
base_colormaps = ['cool', 'Purples', 'spring', 'Blues', 'Oranges', 'Greens', 'plasma', 'Wistia', 'Reds', 'autumn']

for j in range(n_plot):
    points = np.array([X[:, j], Y[:, j]]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    time_weights = np.linspace(0.1, 1.0, X.shape[0])
    cmap_sel = plt.get_cmap(base_colormaps[j % len(base_colormaps)])
    colors = cmap_sel(time_weights)
    lc = LineCollection(segments, colors=colors[:-1], linewidths=1.5, zorder=2)
    axes3.add_collection(lc)
    axes3.scatter(X[:, j], Y[:, j], color=colors, s=8, alpha=time_weights * 0.4, zorder=3)
    axes3.scatter(X[-1, j], Y[-1, j], color=colors[-1], s=35, edgecolor='white', lw=1.0, zorder=5)

# 3. Render the Global Order Parameter Center-of-Mass Vector R(t)
for t in range(0, t_slice - 1, max(1, t_slice // 40)):
    alpha_v = (t / t_slice) * 0.3
    axes3.arrow(0, 0, np.cos(r_angle[t]) * r_mag[t], np.sin(r_angle[t]) * r_mag[t],
             head_width=0.02, head_length=0.02, fc='#FF007F', ec='#FF007F', alpha=alpha_v, zorder=4)

axes3.arrow(0, 0, np.cos(r_angle[-1]) * r_mag[-1], np.sin(r_angle[-1]) * r_mag[-1],
         head_width=0.04, head_length=0.04, fc='#FF007F', ec='#FF007F',
         lw=2.5, zorder=6, label=r"Order Parameter Vector $\mathbf{R}(t_{final})$")

axes3.set_xlim(-1.2, 1.2)
axes3.set_ylim(-1.2, 1.2)
axes3.set_aspect('equal')
axes3.axis('off')

axes3.set_title("Coordinate Boundary Stability Stress Test\n[Time-Dependent Geometric Vector Fields]",
             fontsize=13, fontweight="bold", pad=20)
axes3.legend(loc="upper right", frameon=True, facecolor='#121212', edgecolor='#2A2A2A')
_apply_aclonica_font(axes3)

plt.tight_layout()
plt.show()

## 4️⃣ Failure Mode 4: Phase Space Distortion (Integrator-Induced Dissipation)

This test measures how faithfully the splitting handles the coupled perturbation to an otherwise conservative free flow.

In [ ]:
# Baseline Kuramoto RHS function for Scipy RK45
def classical_kuramoto_rhs(t, th, omegas, K, N):
    d = th[:, None] - th[None, :]
    return omegas + (K / N) * np.sum(np.sin(-d), axis=1)

def calculate_global_order_parameter(phases):
    return np.abs(np.mean(np.exp(1j * phases), axis=1))

t_eval_sync = np.linspace(0.0, 30.0, 300)

# Instantiate baseline solver for shared state
s_ctl = RotorSolver(n_oscillators=CONFIG.n_osc, dim=2, seed=CONFIG.seed)
init_phases_sync = s_ctl.extract_phases()
intrinsic_omegas = s_ctl.omegas.copy()

# FM-9: K=0 Parity Control (at K=0 flow is theta_i(t) = theta_i(0) + omega_i t)
_, p_ctl = s_ctl.simulate(K=0.0, t_eval=t_eval_sync, dt_internal=0.01, method="geometric")
sol_ctl = solve_ivp(
    classical_kuramoto_rhs, (0.0, 30.0), init_phases_sync,
    method="RK45", t_eval=t_eval_sync,
    args=(intrinsic_omegas, 0.0, CONFIG.n_osc), rtol=1e-10, atol=1e-12,
)

parity_k0 = float(np.max(np.abs(
    np.exp(1j * p_ctl) - np.exp(1j * sol_ctl.y.T)
)))
print(f"[Control] K=0 parity error: {parity_k0:.3e}")

# FM-10: RK45 self-consistency check under tolerance change
def _rk45_end_state(rtol: float) -> np.ndarray:
    s = solve_ivp(
        classical_kuramoto_rhs, (0.0, 30.0), init_phases_sync,
        method="RK45", t_eval=t_eval_sync,
        args=(intrinsic_omegas, CONFIG.K_active, CONFIG.n_osc),
        rtol=rtol, atol=rtol * 1e-2,
    )
    return s.y[:, -30:].T

self_consistency = float(np.sqrt(np.mean(
    (calculate_global_order_parameter(_rk45_end_state(1e-8))
     - calculate_global_order_parameter(_rk45_end_state(1e-12))) ** 2
)))
print(f"[Diagnostic] RK45 self-consistency (rtol 1e-8 vs 1e-12): {self_consistency:.3e}")

In [ ]:
print("👟 Running Failure Mode 4: Critical Coupling Constant (Kc) Sweep...")

coupling_strengths = np.linspace(0.0, 1.5, 15)
order_param_rotor = []
order_param_rk45 = []

# FM-7: Fixed comparison window using matched t_eval grid and tight tolerances
for K in coupling_strengths:
    # 1. Rotor Solver run
    s_loop = RotorSolver(n_oscillators=CONFIG.n_osc, dim=2, seed=CONFIG.seed)
    _, p_rotor = s_loop.simulate(K=K, t_eval=t_eval_sync, dt_internal=0.01, method="geometric")
    order_param_rotor.append(np.mean(calculate_global_order_parameter(p_rotor[-30:])))

    # 2. Scipy RK45 Control Group run with explicit t_eval grid matching rotor solver
    sol_rk = solve_ivp(
        classical_kuramoto_rhs,
        (0.0, 30.0),
        init_phases_sync,
        method="RK45",
        t_eval=t_eval_sync,
        args=(intrinsic_omegas, K, CONFIG.n_osc),
        rtol=1e-10,
        atol=1e-12,
    )
    last_phases_rk = sol_rk.y[:, -30:].T
    order_param_rk45.append(np.mean(calculate_global_order_parameter(last_phases_rk)))

# FM-11: Sigmoid fit for transition width and critical coupling Kc
def sigmoid_transition(K, R_min, R_max, K_c, width):
    return R_min + (R_max - R_min) / (1.0 + np.exp(-(K - K_c) / np.maximum(width, 1e-6)))

popt_rotor, _ = curve_fit(
    sigmoid_transition,
    coupling_strengths,
    order_param_rotor,
    p0=[0.3, 0.9, 0.8, 0.2],
    bounds=([0.0, 0.5, 0.0, 0.001], [0.6, 1.0, 2.0, 2.0]),
    maxfev=10000,
)
popt_rk45, _ = curve_fit(
    sigmoid_transition,
    coupling_strengths,
    order_param_rk45,
    p0=[0.3, 0.9, 0.8, 0.2],
    bounds=([0.0, 0.5, 0.0, 0.001], [0.6, 1.0, 2.0, 2.0]),
    maxfev=10000,
)

transition_width_rotor = popt_rotor[3]
transition_width_rk45 = popt_rk45[3]
kc_rotor = popt_rotor[2]
kc_rk45 = popt_rk45[2]
width_rel_diff = abs(transition_width_rotor - transition_width_rk45) / max(transition_width_rk45, 1e-6)

print(f"    [Transition Width Fit (GA Rotor)]: Width = {transition_width_rotor:.4f} | Kc = {kc_rotor:.4f}")
print(f"    [Transition Width Fit (RK45 Base)]: Width = {transition_width_rk45:.4f} | Kc = {kc_rk45:.4f}")
print(f"    [Relative Transition Width Diff]:   {width_rel_diff:.4f}")

# Plot results
figure, axes4 = plt.subplots(figsize=(8, 5))
K_smooth = np.linspace(0.0, 1.5, 200)

axes4.plot(coupling_strengths, order_param_rotor, 'o', color=SCRIBER_PALETTE["accent_blue"], label="GA Rotor Solver (Data)")
axes4.plot(K_smooth, sigmoid_transition(K_smooth, *popt_rotor), '-', color=SCRIBER_PALETTE["accent_blue"], lw=2.0, label=f"GA Rotor Fit (w={transition_width_rotor:.3f})")
axes4.plot(coupling_strengths, order_param_rk45, '--x', color=SCRIBER_PALETTE["accent_orange"], label="Classical RK45 Base (Data)")
axes4.plot(K_smooth, sigmoid_transition(K_smooth, *popt_rk45), ':', color=SCRIBER_PALETTE["accent_orange"], lw=2.0, label=f"RK45 Base Fit (w={transition_width_rk45:.3f})")
axes4.set_title(r"Critical Coupling Phase Transition ($K_c$ Sweep & Sigmoid Fits)", fontsize=13, fontweight="bold")
axes4.set_xlabel("Coupling Strength ($K$)", fontsize=11)
axes4.set_ylabel(r"Steady-State Order Parameter $\langle R \rangle$", fontsize=11)
axes4.legend()
axes4.grid(True, alpha=0.1)
_apply_aclonica_font(axes4)

plt.tight_layout()
plt.show()

## 5️⃣ Failure Report - CI-Ready Pass/Fail Aggregation

In [ ]:
import json
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

Direction = Literal["below", "above"]

# ---------------------------------------------------------------------------
# Calibrated thresholds with explicit physical / numerical justifications:
# ---------------------------------------------------------------------------
TOL_MANIFOLD_DRIFT = 1e-10   # max |R R† - 1|; RKMK4 sits at round-off floor (~1e-12), threshold is ~100x above floor
TOL_ALIAS_RATIO    = 1.0     # PSD(dt) / PSD(dt/2) in upper band; ratio < 1 confirms no spurious energy injection
TOL_VELOCITY_SPIKE = 50.0    # max |dθ/dt| as K -> 0 [rad/s]; unwrapped max spike is ~0.97 rad/s
TOL_PARITY_K0      = 1e-10   # K=0 flow exact match; observed parity error is ~1.8e-12
TOL_PARITY_RMSE    = 0.05    # RMS |<R>_rotor - <R>_rk45| across K sweep; observed RMSE is ~0.0010


@dataclass(frozen=True)
class FailureCheck:
    """
    A single falsifiable assertion about integration behavior.

    Each check is a claim the solver is permitted to *fail*. Silence is not evidence: a mode that cannot fail is not a test.
    """

    mode: str
    name: str
    value: float
    threshold: float
    direction: Direction = "below"
    units: str = ""
    note: str = ""

    @property
    def passed(self) -> bool:
        if self.direction == "below":
            return self.value <= self.threshold
        return self.value >= self.threshold


CHECKS: list[FailureCheck] = []

# SHARED-2: Windows-safe report timestamp (ISO-8601 basic format, no colons)
def _safe_stamp() -> str:
    """ISO-8601 basic format. No colons: filesystem-safe on Windows."""
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def render_failure_report(
    checks: list[FailureCheck],
    *,
    out_dir: Path | None = Path("reports"),
    verbose: bool = True,
) -> dict:
    """
    Aggregate checks, print a table, persist JSON, return a summary dict.

    Parameters
    ----------
    out_dir : Path
        output directory for JSON report
    verbose : bool
        print a longer failure mode report

    Returns
    -------
    A summary dict of failure modes.
    """
    total = len(checks)
    failed = [c for c in checks if not c.passed]

    if verbose:
        print("\n" + "=" * 76)
        print(f"{'MODE':<24}{'CHECK':<24}{'VALUE':>12}{'LIMIT':>12}  STATUS")
        print("-" * 76)
        for c in checks:
            print(
                f"{c.mode:<24}{c.name:<24}"
                f"{c.value:>12.3e}{c.threshold:>12.3e}    "
                f"{'PASS ✅' if c.passed else 'FAIL ❌'}"
            )
        print("-" * 76)
        print(f"{total - len(failed)}/{total} checks passed")
        for c in failed:
            print(
                f"  FAIL  [{c.mode}] {c.name}: {c.value:.3e} "
                f"(limit {c.threshold:.3e})  {c.note}"
            )

    summary = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "total": total,
        "passed": total - len(failed),
        "failed": len(failed),
        "ok": not failed,
        "checks": [asdict(c) | {"passed": c.passed} for c in checks],
    }

    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"failure_report_{_safe_stamp()}.json"
        path.write_text(json.dumps(summary, indent=2))
        if verbose:
            print(f"Report written to {path}")

    return summary


def assert_all_passed(summary: dict) -> None:
    """CI hook: raise so the pipeline exits non-zero on any regression."""
    if not summary["ok"]:
        bad = ", ".join(
            f"{c['mode']}/{c['name']}"
            for c in summary["checks"]
            if not c["passed"]
        )
        raise AssertionError(
            f"{summary['failed']} failure-mode checks tripped: {bad}"
        )

In [ ]:
# --- Mode 1: geometric denormalization -------------------------------------
CHECKS.append(FailureCheck(
    mode="denormalization",
    name="max_manifold_drift",
    value=float(manifold_drift.max()),
    threshold=TOL_MANIFOLD_DRIFT,
    units="dimensionless",
    note="RKMK4 should sit at round-off (~1e-12); threshold 1e-10 is ~100x above floor.",
))

In [ ]:
# --- Mode 2: spurious high-frequency energy pumping ------------------------
CHECKS.append(FailureCheck(
    mode="aliasing",
    name="hf_psd_ratio",
    value=alias_ratio,
    threshold=TOL_ALIAS_RATIO,
    note="Mean upper band PSD ratio; ratio < 1.0 confirms halving dt reduced high-frequency power.",
))
# FM-3: worst-case oscillator PSD ratio
CHECKS.append(FailureCheck(
    mode="aliasing",
    name="hf_psd_ratio_worst_case",
    value=alias_ratio_max,
    threshold=TOL_ALIAS_RATIO,
    note="Worst-case oscillator PSD ratio in upper band.",
))

In [ ]:
# --- Mode 3: coordinate singularity blindspots -----------------------------
CHECKS.append(FailureCheck(
    mode="singularity",
    name="nan_free",
    value=1.0 if has_nan_failure else 0.0,
    threshold=0.0,
    note="Any NaN at K -> 0 is a hard failure.",
))
CHECKS.append(FailureCheck(
    mode="singularity",
    name="max_velocity_spike",
    value=float(max_velocity_spike),
    threshold=TOL_VELOCITY_SPIKE,
    units="rad/s",
    note="Unwrapped phase differentiation velocity spike (threshold 50 rad/s vs observed ~0.97 rad/s).",
))

In [ ]:
# --- Mode 4: phase space distortion & controls -----------------------------
# FM-9: K=0 Parity Control check
CHECKS.append(FailureCheck(
    mode="parity_control",
    name="k0_parity",
    value=parity_k0,
    threshold=TOL_PARITY_K0,
    note="Decoupled K=0 flow exact match against analytical / RK45 reference.",
))

# FM-7: Phase space distortion parity RMSE
parity_rmse = float(np.sqrt(np.mean(
    (np.asarray(order_param_rotor) - np.asarray(order_param_rk45)) ** 2
)))
CHECKS.append(FailureCheck(
    mode="phase_space_distortion",
    name="rk45_parity_rmse",
    value=parity_rmse,
    threshold=TOL_PARITY_RMSE,
    note="Curve-shape agreement across K sweep sampled on identical grid.",
))

In [ ]:
# --- Aggregate -------------------------------------------------------------
report = render_failure_report(CHECKS)
# assert_all_passed(report)   # uncomment in CI / pre-commit
